# Download Microsoft Building Polygon Tiles

Overview of the Code

This Python script downloads and processes building footprint data for specific geographic tiles that intersect a predefined Area of Interest (AOI). It does so efficiently using parallel processing with ProcessPoolExecutor. The workflow is broken into multiple steps:

1. Configuration Variables

    Defines file paths for the shapefile, dataset, and output directory.
    Specifies processing parameters like zoom level (9), buffer distance (0.01 degrees), and parallel processing limits (40 workers).

2. Load and Process AOI (Area of Interest)

    Loads a shapefile (.shp) that defines the study area.
    Converts it to the appropriate coordinate system (EPSG:4326).
    Expands the area slightly with a buffer to capture nearby buildings.

3. Identify Tiles Intersecting the AOI

    Uses mercantile to generate map tiles that cover the AOI at zoom level 9.
    Filters the tiles to include only those that intersect the buffered AOI.

4. Plot the AOI and Intersecting Tiles

    Uses geopandas and matplotlib to visualize:
        The buffered AOI boundary (blue).
        The intersecting tile boundaries (red dashed lines).

5. Sample Tiles for Testing

    Randomly selects 2% of tiles for testing before processing everything.

6. Download Dataset and Filter QuadKeys

    Reads a CSV file containing dataset links.
    Filters out quad keys (tile identifiers) that do not exist in the dataset.
    Saves a list of tiles with multiple dataset entries to a CSV.

7. Check Existing Downloads

    Scans the output directory for already-downloaded files.
    Determines which quad keys still need to be processed.

8. Define Parallel Download & Processing Function

    A function download_and_process_tile():
        Fetches JSON data containing building footprints for a given quad key.
        Converts JSON to a GeoDataFrame.
        Filters footprints to those inside the AOI.
        Saves the resulting dataset in Feather format for efficient storage.

9. Process Tiles in Parallel

    Uses ProcessPoolExecutor to download and process tiles in parallel.
    Displays progress using tqdm progress bar.

In [ ]:
import os
WRI_PROJECT_ROOT = os.environ.get("WRI_PROJECT_ROOT", "/home/shares/wwri-wildfire")

import pandas as pd
import geopandas as gpd
from shapely import geometry
import mercantile
from tqdm import tqdm
import os
import time
from concurrent.futures import ProcessPoolExecutor, as_completed
import matplotlib.pyplot as plt
import random

# ==============================
# Configuration Variables
# ==============================

# File Paths
SHAPEFILE_PATH = os.path.join(WRI_PROJECT_ROOT, 'data', 'multi-domain-data', 'boundary-layers', 'processed', 'admin-boundary-layers', 'wwri_study_area_admin_0.shp')
DATASET_LINKS_CSV_URL = "https://minedbuildings.z5.web.core.windows.net/global-buildings/dataset-links.csv"

# Output Configuration
OUTPUT_DIR = os.path.join(WRI_PROJECT_ROOT, 'data', 'built-environment-domain-data', 'defensible-space', 'structure-polygons', 'zoom_9_raw_structure_polygons')
MULTIPLE_ENTRIES_CSV = os.path.join(OUTPUT_DIR, "multiple_entries_quad_keys.csv")
OUTPUT_FEATHER_FILENAME = "building_footprints.feather"  # Not used per tile, kept for consistency

# Processing Parameters
EPSG_CODE = "EPSG:4326"
BUFFER_DISTANCE_DEGREES = 0.01  # Approximately 1 km at the equator
ZOOM_LEVEL = 9
SAMPLING_PERCENTAGE = 0.02
MAX_WORKERS = 40

# ==============================
# Step 1 - Define Area of Interest (AOI)
# ==============================
print("Loading the shapefile...")
aoi_gdf = gpd.read_file(SHAPEFILE_PATH)

# Ensure the AOI is in the specified EPSG
print(f"Ensuring AOI is in {EPSG_CODE}...")
if aoi_gdf.crs != EPSG_CODE:
    aoi_gdf = aoi_gdf.to_crs(EPSG_CODE)

# Assuming the shapefile has only one polygon, otherwise select the specific polygon
aoi_shape = aoi_gdf.geometry.iloc[0]

# Apply a buffer around the AOI
print(f"Applying buffer of {BUFFER_DISTANCE_DEGREES} degrees around the AOI...")
aoi_shape = aoi_shape.buffer(BUFFER_DISTANCE_DEGREES)

# ==============================
# Step 2 - Determine Intersecting Tiles
# ==============================
print("Determining which tiles intersect the AOI...")

def tile_intersects_polygon(tile, polygon):
    tile_bounds = mercantile.bounds(tile)
    tile_polygon = geometry.box(tile_bounds.west, tile_bounds.south, tile_bounds.east, tile_bounds.north)
    return tile_polygon.intersects(polygon)

tile_polygons = []
quad_keys = set()

# Generate tiles within AOI bounds at specified zoom level
tiles = mercantile.tiles(
    west=aoi_shape.bounds[0],
    south=aoi_shape.bounds[1],
    east=aoi_shape.bounds[2],
    north=aoi_shape.bounds[3],
    zooms=ZOOM_LEVEL
)

for tile in list(tiles):
    if tile_intersects_polygon(tile, aoi_shape):
        quad_keys.add(mercantile.quadkey(tile))
        tile_polygons.append(geometry.box(*mercantile.bounds(tile)))

quad_keys = list(quad_keys)
print(f"The input area spans {len(quad_keys)} tiles: {quad_keys}")

# ==============================
# Plot AOI and Intersecting Tiles
# ==============================
print("Plotting the AOI and intersecting tiles...")
aoi_gdf_buffered = gpd.GeoDataFrame(geometry=[aoi_shape], crs=EPSG_CODE)
tile_gdf = gpd.GeoDataFrame(geometry=tile_polygons, crs=EPSG_CODE)

fig, ax = plt.subplots(figsize=(10, 10))
aoi_gdf_buffered.boundary.plot(ax=ax, color='blue', label='Buffered AOI')
tile_gdf.boundary.plot(ax=ax, color='red', linestyle='--', label='Intersecting Tiles')
plt.legend()
plt.title("AOI and Intersecting Tiles")
plt.show()

# ==============================
# Step 3 - Sample Tiles for Testing
# ==============================
sampling_percentage = SAMPLING_PERCENTAGE
num_tiles_to_process = max(1, int(len(quad_keys) * sampling_percentage))
quad_keys_sampled = random.sample(quad_keys, num_tiles_to_process)
print(f"Processing a random sample of {num_tiles_to_process} tiles for testing.")

# ==============================
# Step 4 - Download Building Footprints
# ==============================
print("Reading dataset links CSV...")
df = pd.read_csv(DATASET_LINKS_CSV_URL, dtype=str)
print("Dataset links CSV read successfully.")

# Filter out quad keys not present in the dataset
dataset_quad_keys = set(df["QuadKey"].unique())
quad_keys_sampled = [quad_key for quad_key in quad_keys_sampled if quad_key in dataset_quad_keys]
print(f"Filtered quad keys, {len(quad_keys_sampled)} remaining after filtering.")

# Create output directory if it doesn't exist
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Function to calculate folder size
def get_folder_size(folder):
    total_size = 0
    for dirpath, dirnames, filenames in os.walk(folder):
        for f in filenames:
            fp = os.path.join(dirpath, f)
            if os.path.isfile(fp):
                total_size += os.path.getsize(fp)
    return total_size

# Calculate and display the total size of data in the output directory
total_size_gb = get_folder_size(OUTPUT_DIR) / (1024 * 1024 * 1024)
print(f"The folder '{OUTPUT_DIR}' contains approximately {total_size_gb:.2f} GB of data.")

# Identify existing downloaded files
existing_files = set(f.split('.')[0] for f in os.listdir(OUTPUT_DIR) if f.endswith('.feather'))
print(f"Found {len(existing_files)} existing files in the download directory.")

# Determine which quad keys need to be processed
remaining_quad_keys = [quad_key for quad_key in quad_keys_sampled if quad_key not in existing_files]
print(f"Need to process {len(remaining_quad_keys)} remaining tiles.")

# Identify quad keys with multiple entries
multiple_entries_quad_keys = df.groupby("QuadKey").filter(lambda x: len(x) > 1)["QuadKey"].unique()
print(f"Identified {len(multiple_entries_quad_keys)} tiles with multiple entries.")

# Save the list of merged quad keys to a CSV file within OUTPUT_DIR
multiple_entries_quad_keys_df = pd.DataFrame(multiple_entries_quad_keys, columns=["QuadKey"])
multiple_entries_quad_keys_df.to_csv(MULTIPLE_ENTRIES_CSV, index=False)
print(f"Saved the list of tiles with multiple entries to '{MULTIPLE_ENTRIES_CSV}'.")

# ==============================
# Step 5 - Define Download and Processing Function
# ==============================
def download_and_process_tile(quad_key, output_dir, aoi_shape):
    print(f"Processing tile with QuadKey: {quad_key}...")
    fn = os.path.join(output_dir, f"{quad_key}.feather")
    
    if os.path.exists(fn):
        print(f"Tile {quad_key} already exists. Skipping download.")
        return fn
    
    try:
        start_time = time.time()
        rows = df[df["QuadKey"] == quad_key]
        
        if rows.shape[0] > 0:  # Handles both single and multiple entries
            combined_gdf = gpd.GeoDataFrame(columns=["geometry"], crs=EPSG_CODE)
            
            for _, row in rows.iterrows():
                url = row["Url"]
                df2 = pd.read_json(url, lines=True)
                df2["geometry"] = df2["geometry"].apply(geometry.shape)
                gdf = gpd.GeoDataFrame(df2, crs=EPSG_CODE)
                gdf_within_aoi = gdf[gdf.geometry.within(aoi_shape)]
                combined_gdf = pd.concat([combined_gdf, gdf_within_aoi], ignore_index=True)
            
            if not combined_gdf.empty:
                combined_gdf.reset_index(drop=True).to_feather(fn)
                file_size = os.path.getsize(fn) / (1024 * 1024)  # Convert bytes to megabytes
                end_time = time.time()
                print(f"Tile {quad_key} processed and saved successfully in {end_time - start_time:.2f} seconds. File size: {file_size:.2f} MB.")
                return fn
            else:
                print(f"No building footprints within AOI for tile {quad_key}.")
                return None
        else:
            print(f"QuadKey not found in dataset: {quad_key}")
            return None
    except Exception as e:
        print(f"Error processing tile {quad_key}: {e}")
        return None

# ==============================
# Step 6 - Download and Process Tiles in Parallel
# ==============================
downloaded_files = []
print("Downloading and processing tiles...")

with ProcessPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {
        executor.submit(download_and_process_tile, quad_key, OUTPUT_DIR, aoi_shape): quad_key
        for quad_key in remaining_quad_keys
    }
    
    with tqdm(total=len(futures)) as pbar:
        for future in as_completed(futures):
            quad_key = futures[future]
            try:
                fn = future.result()
                if fn is not None:
                    downloaded_files.append(fn)
            except Exception as e:
                print(f"Tile {quad_key} encountered an error: {e}")
            finally:
                pbar.update(1)

print("All tiles downloaded and processed.")
